# Online reboot

In [ ]:
import time
import cflib.crtp
from cflib.utils.power_switch import PowerSwitch

uri = "radio://0/90/2M/E7E7E7E705"

cflib.crtp.init_drivers()

ps = None

try:
    ps = PowerSwitch(uri)

    print("Reiniciando Crazyflie Bolt...")
    ps.stm_power_cycle()

    time.sleep(3)

    print("Reinicio completado")

except Exception as e:
    print(f"Error: {e}")

finally:
    if ps:
        ps.close()

Reiniciando Crazyflie Bolt...
Reinicio completado


In [ ]:
#!/usr/bin/env python3
import logging
import time
from collections import deque

import cflib.crtp
from cflib.crazyflie import Crazyflie
from cflib.crazyflie.syncCrazyflie import SyncCrazyflie


URI = "radio://0/90/2M/E7E7E7E705"

latest = {
    "latency_p95_ms": None,
    "link_quality": None,
    "uplink_rssi": None,
    "uplink_rate": None,
    "downlink_rate": None,
    "uplink_congestion": None,
    "downlink_congestion": None,
}

history = deque(maxlen=50)
best = {
    "latency_p95_ms": float("inf"),
    "link_quality": None,
    "uplink_rssi": None,
    "time": None,
}


def set_value(key):
    def callback(value):
        latest[key] = value
    return callback


def latency_callback(latency_ms):
    latency_ms = float(latency_ms)
    latest["latency_p95_ms"] = latency_ms
    history.append(latency_ms)

    if latency_ms < best["latency_p95_ms"]:
        best["latency_p95_ms"] = latency_ms
        best["link_quality"] = latest["link_quality"]
        best["uplink_rssi"] = latest["uplink_rssi"]
        best["time"] = time.strftime("%H:%M:%S")


def fmt(value, decimals=1):
    if value is None:
        return "--"
    if isinstance(value, float):
        return f"{value:.{decimals}f}"
    return str(value)


def bar_from_latency(lat_ms):
    if lat_ms is None:
        return "----------"

    # Ajusta estos umbrales según tu entorno.
    # Menor latencia = mejor posición.
    if lat_ms < 8:
        return "██████████ EXCELENTE"
    if lat_ms < 15:
        return "████████-- BUENA"
    if lat_ms < 30:
        return "█████----- REGULAR"
    return "██-------- MALA"


def main():
    logging.basicConfig(level=logging.ERROR)

    cflib.crtp.init_drivers()

    print(f"Conectando a {URI}")
    print("Cierra cfclient si está usando el mismo Crazyradio.\n")

    with SyncCrazyflie(URI, cf=Crazyflie(rw_cache="./cache")) as scf:
        cf = scf.cf
        stats = cf.link_statistics

        stats.latency_updated.add_callback(latency_callback)

        # Estas métricas dependen del driver radio y de la versión de cflib.
        stats.link_quality_updated.add_callback(set_value("link_quality"))
        stats.uplink_rssi_updated.add_callback(set_value("uplink_rssi"))
        stats.uplink_rate_updated.add_callback(set_value("uplink_rate"))
        stats.downlink_rate_updated.add_callback(set_value("downlink_rate"))
        stats.uplink_congestion_updated.add_callback(set_value("uplink_congestion"))
        stats.downlink_congestion_updated.add_callback(set_value("downlink_congestion"))

        print("Conectado. Mueve la antena lentamente y observa la latencia.")
        print("Ctrl+C para salir.\n")

        try:
            while True:
                lat = latest["latency_p95_ms"]

                if len(history) > 0:
                    avg = sum(history) / len(history)
                else:
                    avg = None

                line = (
                    f"\rLat p95: {fmt(lat):>6} ms | "
                    f"prom: {fmt(avg):>6} ms | "
                    f"LQ: {fmt(latest['link_quality']):>5} | "
                    f"RSSI up: {fmt(latest['uplink_rssi']):>6} | "
                    f"Best: {fmt(best['latency_p95_ms']):>6} ms "
                    f"@ {best['time'] or '--'} | "
                    f"{bar_from_latency(lat):<22}"
                )

                print(line, end="", flush=True)
                time.sleep(0.2)

        except KeyboardInterrupt:
            print("\n\nFinalizado.")
            print(
                f"Mejor latencia p95: {best['latency_p95_ms']:.1f} ms "
                f"@ {best['time']}"
            )
            print(f"Link quality en mejor punto: {best['link_quality']}")
            print(f"RSSI uplink en mejor punto: {best['uplink_rssi']}")


if __name__ == "__main__":
    main()

Conectando a radio://0/90/2M/E7E7E7E705
Cierra cfclient si está usando el mismo Crazyradio.

Conectado. Mueve la antena lentamente y observa la latencia.
Ctrl+C para salir.

Lat p95:    5.9 ms | prom:    5.5 ms | LQ:  99.5 | RSSI up:     -- | Best:    4.0 ms @ 13:53:23 | ██████████ EXCELENTE  